# Chunking Analysis for `raw_dataset.jsonl`

This notebook studies how to transform `artifacts/raw_dataset.jsonl` into indexable context chunks for dense and sparse Qdrant indexing.

The central question is: **what is the smallest coherent evidence unit that should be retrievable and sent to an LLM?**

For this dataset, raw rows contain ordered `candidates` extracted from source pages. Some candidates are complete paragraphs, but many are headers, list introductions, table fragments, boilerplate, or continuation blocks. This notebook analyzes those patterns and uses the results to propose a chunking strategy.


## Foundations: What Chunking Is

Chunking converts raw source material into retrieval units. In this project, the target dataflow should be:

```text
HF dataset
  -> artifacts/raw_dataset.jsonl        # source-of-truth rows
  -> artifacts/index_chunks.jsonl       # processed retrieval units
  -> Qdrant dense + sparse vectors
```

A chunk becomes one Qdrant point. Chunk design controls what gets embedded for dense retrieval, tokenized for sparse retrieval, returned as evidence, and ultimately shown to the LLM.

Bad chunking creates bad context. For example, this standalone candidate is incomplete:

```text
Kolkata is the magic capital of India and has produced internationally famous magicians and performers, including:
```

The adjacent candidate may contain the actual answer body:

```text
P.C. Sorcar
P.C. Sorcar, Jr.
P.C. Sorcar, Young
```

The coherent chunk is the merged span.


## Best Practices and Common Approaches

### Fixed-size token chunking
Split every document into fixed token windows, often with overlap. This is simple and scalable but ignores page structure and may split tables, lists, or sentence continuations.

### Sentence or paragraph chunking
Use natural language boundaries. This works well for prose but less well for HTML/table/list-derived candidates.

### Structure-aware chunking
Use source structure such as headers, list blocks, table rows, and candidate order. This preserves meaning best for this dataset, but it requires dataset-specific heuristics.

### Semantic chunking
Use embeddings or topic-shift detection to split and merge content. This can adapt well but is harder to reproduce and expensive for a first pass.

### Hierarchical retrieval
Retrieve smaller chunks but keep links to parent groups for expansion. This gives good precision with context recovery but requires richer metadata and retrieval-time logic.

## Working Hypothesis

For `sentence-transformers/NQ-retrieval`, v1 should use **structure-aware candidate span chunking**: preserve row order, clean/drop boilerplate, merge adjacent candidates that form evidence units, split only when too large, and write one chunk artifact consumed by both dense and sparse indexers.


In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from statistics import mean, median
from typing import Any

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').is_file():
    # Notebook may be launched from src/scripts.
    REPO_ROOT = Path.cwd().parents[1]

RAW_PATH = REPO_ROOT / 'artifacts' / 'raw_dataset.jsonl'
REPORT_PATH = REPO_ROOT.parent / '.cursor' / 'rag-nq-showcase' / 'chunking_strategy.md'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

MAX_ROWS = 5000
MAX_EXAMPLES = 8

print('repo:', REPO_ROOT)
print('raw artifact:', RAW_PATH)
print('strategy report:', REPORT_PATH)


## Dataset Loading

This notebook expects `artifacts/raw_dataset.jsonl` to exist. If it does not, create it first:

```bash
python -m src.ingestion.ingest_raw
```

For quick iteration, set `RAG_MAX_PASSAGES` before raw ingest to create a capped sample.


In [ ]:
def load_raw_rows(path: Path, max_rows: int | None = None) -> list[dict[str, Any]]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. Run: python -m src.ingestion.ingest_raw')
    rows: list[dict[str, Any]] = []
    with path.open('r', encoding='utf-8') as handle:
        for line_no, line in enumerate(handle, start=1):
            if max_rows is not None and len(rows) >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if not isinstance(obj, dict):
                raise ValueError(f'Line {line_no} is not a JSON object')
            rows.append(obj)
    return rows

rows = load_raw_rows(RAW_PATH, MAX_ROWS)
print(f'Loaded {len(rows):,} rows for analysis')
print('First row keys:', sorted(rows[0].keys()) if rows else [])


## Helper Functions

These helpers are intentionally simple and reproducible. The notebook measures enough of the dataset to justify a v1 chunking strategy; it is not final production chunker code.


In [ ]:
_whitespace_re = re.compile(r'\s+')

BOILERPLATE_EXACT = {'vte', 'v t e', 'v\nt\ne', 'external links', 'references', 'see also', 'notes'}
INTRO_MARKERS = ('including', 'include', 'includes', 'such as', 'the following', 'for example', 'legend')
LIST_OR_TABLE_TYPES = {'list', 'table'}
HEADER_TYPES = {'list_definition', 'section', 'header'}


def normalize_text(text: object) -> str:
    s = str(text).replace('\u00a0', ' ').strip()
    return _whitespace_re.sub(' ', s)


def display_text(text: str, max_len: int = 220) -> str:
    text = text.replace('\n', '\\n')
    return text if len(text) <= max_len else text[:max_len] + '...'


def token_count(text: str) -> int:
    return len(re.findall(r'\w+', text))


def is_boilerplate(text: str) -> bool:
    n = normalize_text(text).casefold()
    return n in BOILERPLATE_EXACT or len(n) <= 2


def looks_like_intro(text: str) -> bool:
    n = normalize_text(text).casefold()
    if not n:
        return False
    if n.endswith(':'):
        return True
    return any(marker in n for marker in INTRO_MARKERS) and len(n.split()) <= 40


def looks_like_list_or_table_body(text: str, passage_type: str | None) -> bool:
    if passage_type in LIST_OR_TABLE_TYPES:
        return True
    return '\n' in str(text) and token_count(str(text)) >= 2


def is_short_header(text: str, passage_type: str | None) -> bool:
    if passage_type in HEADER_TYPES:
        return True
    return token_count(text) <= 8 and normalize_text(text).endswith(':')


def row_candidates(row: dict[str, Any]) -> list[str]:
    cands = row.get('candidates')
    if not isinstance(cands, list):
        return []
    return [str(c).strip() for c in cands if str(c).strip()]


def row_passage_types(row: dict[str, Any]) -> list[str | None]:
    types = row.get('passage_types')
    if not isinstance(types, list):
        return []
    return [str(t).strip() if t is not None else None for t in types]


## Analysis 1: Raw Shape and Candidate Volume

This tells us whether candidate-level indexing creates too many tiny points and whether row-level concatenation creates oversized points.


In [ ]:
row_candidate_counts = [len(row_candidates(row)) for row in rows]
all_candidates = []
for row_i, row in enumerate(rows):
    types = row_passage_types(row)
    for cand_i, text in enumerate(row_candidates(row)):
        passage_type = types[cand_i] if cand_i < len(types) else None
        all_candidates.append((row_i, cand_i, text, passage_type, row))

candidate_token_counts = [token_count(text) for _, _, text, _, _ in all_candidates]
row_token_counts = [sum(token_count(c) for c in row_candidates(row)) for row in rows]

summary = {
    'sample_rows': len(rows),
    'total_candidates': len(all_candidates),
    'avg_candidates_per_row': round(mean(row_candidate_counts), 2) if row_candidate_counts else 0,
    'median_candidates_per_row': median(row_candidate_counts) if row_candidate_counts else 0,
    'avg_candidate_tokens': round(mean(candidate_token_counts), 2) if candidate_token_counts else 0,
    'median_candidate_tokens': median(candidate_token_counts) if candidate_token_counts else 0,
    'avg_row_tokens_if_concatenated': round(mean(row_token_counts), 2) if row_token_counts else 0,
    'median_row_tokens_if_concatenated': median(row_token_counts) if row_token_counts else 0,
}
summary


### Interpretation

If candidates are very short, candidate-as-point indexing produces weak fragments. If rows are very large, whole-row concatenation produces low-precision chunks. A useful chunker should sit between these extremes.


## Analysis 2: Passage Type Mix

The `passage_types` field helps identify prose, list content, table content, and header/definition candidates.


In [ ]:
type_counts = Counter(pt or '<missing>' for _, _, _, pt, _ in all_candidates)
type_counts.most_common(20)


## Analysis 3: Orphaned Intros and Adjacent Continuations

This test looks for candidate `i` that looks like an intro/header and candidate `i+1` that looks like list/table/body content. These are direct evidence that one-candidate-one-point indexing breaks context.


In [ ]:
continuation_examples = []
continuation_count = 0
adjacent_pairs = 0

for row_i, row in enumerate(rows):
    cands = row_candidates(row)
    types = row_passage_types(row)
    for i in range(len(cands) - 1):
        left = cands[i]
        right = cands[i + 1]
        left_type = types[i] if i < len(types) else None
        right_type = types[i + 1] if i + 1 < len(types) else None
        adjacent_pairs += 1
        if (looks_like_intro(left) or is_short_header(left, left_type)) and looks_like_list_or_table_body(right, right_type):
            continuation_count += 1
            if len(continuation_examples) < MAX_EXAMPLES:
                continuation_examples.append({
                    'row': row_i,
                    'candidate_span': f'{i}->{i+1}',
                    'title': row.get('title'),
                    'question': row.get('question'),
                    'left_type': left_type,
                    'right_type': right_type,
                    'left': display_text(left),
                    'right': display_text(right),
                })

continuation_summary = {
    'adjacent_pairs': adjacent_pairs,
    'likely_continuations': continuation_count,
    'ratio': round(continuation_count / adjacent_pairs, 4) if adjacent_pairs else 0,
}
continuation_summary, continuation_examples


### Interpretation

A non-trivial continuation count means we should merge **local adjacent spans**. This does not imply concatenating whole rows; it means preserving order to recover evidence units split by source extraction.


## Analysis 4: Boilerplate and Frequent Duplicate Text

Repeated snippets like navigation blocks, infobox labels, and generic section headers can dominate sparse retrieval and pollute dense retrieval.


In [ ]:
normalized_counts = Counter(normalize_text(text).casefold() for _, _, text, _, _ in all_candidates)
frequent = [(text, count) for text, count in normalized_counts.most_common(25) if count >= 2]
boilerplate_count = sum(1 for _, _, text, _, _ in all_candidates if is_boilerplate(text))

{
    'boilerplate_candidates': boilerplate_count,
    'boilerplate_ratio': round(boilerplate_count / len(all_candidates), 4) if all_candidates else 0,
    'top_frequent_text': frequent[:15],
}


### Interpretation

High-frequency, low-information snippets should generally not become standalone points. Some should be dropped before chunking; others should only be retained when attached to nearby meaningful content.


## Analysis 5: Simulated Chunking Policies

We compare three policies:

1. **Candidate-as-chunk**: old behavior; one candidate becomes one point.
2. **Whole-row concat**: entire row becomes one point.
3. **Structure-aware spans**: drop obvious boilerplate, merge adjacent intros/headers with bodies, split when too large.


In [ ]:
@dataclass
class SimChunk:
    row: int
    start: int
    end: int
    text: str
    kinds: list[str | None]


def make_candidate_chunks(rows: list[dict[str, Any]]) -> list[SimChunk]:
    chunks = []
    for row_i, row in enumerate(rows):
        types = row_passage_types(row)
        for i, text in enumerate(row_candidates(row)):
            if is_boilerplate(text):
                continue
            chunks.append(SimChunk(row_i, i, i, text, [types[i] if i < len(types) else None]))
    return chunks


def make_row_chunks(rows: list[dict[str, Any]]) -> list[SimChunk]:
    chunks = []
    for row_i, row in enumerate(rows):
        cands = [c for c in row_candidates(row) if not is_boilerplate(c)]
        if cands:
            chunks.append(SimChunk(row_i, 0, len(cands) - 1, '\n'.join(cands), ['row']))
    return chunks


def make_structure_chunks(rows: list[dict[str, Any]], max_tokens: int = 220) -> list[SimChunk]:
    chunks = []
    for row_i, row in enumerate(rows):
        cands = row_candidates(row)
        types = row_passage_types(row)
        i = 0
        while i < len(cands):
            text = cands[i]
            ptype = types[i] if i < len(types) else None
            if is_boilerplate(text):
                i += 1
                continue
            span_texts = [text]
            span_types = [ptype]
            start = i
            end = i
            while end + 1 < len(cands):
                next_text = cands[end + 1]
                next_type = types[end + 1] if end + 1 < len(types) else None
                if is_boilerplate(next_text):
                    end += 1
                    continue
                merged_text = '\n'.join(span_texts + [next_text])
                should_merge = (
                    looks_like_intro(span_texts[-1])
                    or is_short_header(span_texts[-1], span_types[-1])
                    or (span_types[-1] in HEADER_TYPES and looks_like_list_or_table_body(next_text, next_type))
                )
                if not should_merge or token_count(merged_text) > max_tokens:
                    break
                span_texts.append(next_text)
                span_types.append(next_type)
                end += 1
            chunks.append(SimChunk(row_i, start, end, '\n'.join(span_texts), span_types))
            i = end + 1
    return chunks


def policy_metrics(chunks: list[SimChunk]) -> dict[str, Any]:
    toks = [token_count(c.text) for c in chunks]
    incomplete_intro = [c for c in chunks if looks_like_intro(c.text) and c.start == c.end]
    tiny = [c for c in chunks if token_count(c.text) <= 3]
    huge = [c for c in chunks if token_count(c.text) > 300]
    return {
        'chunks': len(chunks),
        'avg_tokens': round(mean(toks), 2) if toks else 0,
        'median_tokens': median(toks) if toks else 0,
        'tiny_chunks_<=3_tokens': len(tiny),
        'standalone_intro_chunks': len(incomplete_intro),
        'huge_chunks_>300_tokens': len(huge),
    }

candidate_chunks = make_candidate_chunks(rows)
row_chunks = make_row_chunks(rows)
structure_chunks = make_structure_chunks(rows)

simulation = {
    'candidate_as_chunk': policy_metrics(candidate_chunks),
    'whole_row_concat': policy_metrics(row_chunks),
    'structure_aware_spans': policy_metrics(structure_chunks),
}
simulation


## Analysis 6: Example Chunks From Structure-Aware Policy

The examples below show the kind of text that would become one Qdrant point.


In [ ]:
examples = []
for chunk in structure_chunks:
    if chunk.end > chunk.start or token_count(chunk.text) >= 25:
        row = rows[chunk.row]
        examples.append({
            'row': chunk.row,
            'span': f'{chunk.start}-{chunk.end}',
            'title': row.get('title'),
            'question': row.get('question'),
            'types': chunk.kinds,
            'tokens': token_count(chunk.text),
            'text': display_text(chunk.text, 500),
        })
    if len(examples) >= MAX_EXAMPLES:
        break
examples


## Proposed Data Model

The chunking stage should produce a dedicated artifact, not overwrite raw rows.

Suggested artifact:

```text
artifacts/index_chunks.jsonl
```

Each line should represent one indexable Qdrant point:

```python
@dataclass
class IndexChunk:
    chunk_id: str
    group_id: str
    text: str
    title: str | None
    source: str | None
    question: str | None
    document_url: str | None
    source_row_ordinal: int
    start_candidate_idx: int
    end_candidate_idx: int
    passage_types: list[str]
    long_answers: list[str]
    chunk_kind: str
    token_count: int
```

Dense and sparse indexing should consume the same `index_chunks.jsonl` so hybrid results refer to the same Qdrant point IDs.


## Final Strategy Writer

This cell writes `.cursor/rag-nq-showcase/chunking_strategy.md` using the analysis results. The file becomes the implementation guide for the next ingestion/indexing step.


In [ ]:
def fmt_dict(d: dict[str, Any]) -> str:
    return '\n'.join(f'- `{k}`: {v}' for k, v in d.items())

lines = [
    '# Chunking Strategy for `rag-nq-showcase`',
    '',
    '## Recommendation',
    '',
    'Use **structure-aware candidate span chunking** over `artifacts/raw_dataset.jsonl` and write the result to `artifacts/index_chunks.jsonl` before dense/sparse indexing.',
    '',
    'Do not index raw candidates directly. Do not concatenate entire rows blindly.',
    '',
    '## Why',
    '',
    'The dataset contains ordered candidates that often represent page fragments: prose, list introductions, list bodies, table fragments, headers, and boilerplate. Candidate-level indexing can retrieve incomplete evidence, while row-level concatenation creates overly broad chunks.',
    '',
    '## Sample Summary',
    '',
    fmt_dict(summary),
    '',
    '## Continuation Evidence',
    '',
    fmt_dict(continuation_summary),
    '',
    'These continuation pairs support preserving candidate order and merging adjacent spans when candidate `i` introduces candidate `i+1`.',
    '',
    '## Policy Simulation',
    '',
    '### Candidate-as-chunk',
    fmt_dict(simulation['candidate_as_chunk']),
    '',
    '### Whole-row concat',
    fmt_dict(simulation['whole_row_concat']),
    '',
    '### Structure-aware spans',
    fmt_dict(simulation['structure_aware_spans']),
    '',
    '## Final Chunking Rules',
    '',
    '1. Treat each raw row as a `group` and preserve original candidate order.',
    '2. Normalize whitespace and Unicode spacing before analysis/indexing.',
    '3. Drop obvious boilerplate candidates such as `vte`, `v\\nt\\ne`, empty strings, and very short non-informative fragments.',
    '4. Merge adjacent candidates when the left candidate is an intro/header and the right candidate is list/table/body content.',
    '5. Keep complete standalone paragraphs as standalone chunks.',
    '6. Split oversized merged chunks after structure-aware merging, targeting roughly 120-250 tokens per chunk.',
    '7. Prefix table/list splits with useful title/header context when splitting would otherwise lose meaning.',
    '8. Store `group_id`, `source_row_ordinal`, `start_candidate_idx`, `end_candidate_idx`, `passage_types`, `title`, `question`, `document_url`, and `long_answers` in Qdrant payload.',
    '9. Use the same `chunk_id` for dense and sparse vectors in Qdrant.',
    '10. At retrieval time, dedupe by normalized text and cap repeated hits per `group_id`.',
    '',
    '## Proposed Artifacts',
    '',
    '- `artifacts/raw_dataset.jsonl`: raw source-of-truth rows.',
    '- `artifacts/index_chunks.jsonl`: processed chunk artifact consumed by dense and sparse indexing.',
    '- `artifacts/chunk_manifest.json`: schema/config/counts for reproducibility.',
    '',
    '## Qdrant Payload',
    '',
    '- `text`', '- `title`', '- `source`', '- `question`', '- `document_url`', '- `group_id`', '- `source_row_ordinal`', '- `start_candidate_idx`', '- `end_candidate_idx`', '- `passage_types`', '- `chunk_kind`', '- `long_answers` when available',
    '',
    '## Implementation Note',
    '',
    'The chunking stage should be an ingestion module, not part of dense/sparse indexing. Dense and sparse should only read `index_chunks.jsonl` and build vectors over the same chunk IDs.',
]

report = '\n'.join(lines) + '\n'
REPORT_PATH.write_text(report, encoding='utf-8')
print(f'Wrote {REPORT_PATH}')
print(report[:1200])
